# Is a test dataset also a train dataset?

`claude_nest` found no `.geff` under `test/` — but its listing showed `44b6_0b24845f` under
**both** `test/` and `train/`, and all four test stems carry the same `44b6` / `6bba` embryo
prefixes as training.

If that holds, `notes/49`'s "the test set is a third pair of embryos" is wrong, the metric's
`N_est` is readable for part of the test set, and a real local validator becomes possible.

Predictions: (1) a name collision exists; (2) the colliding `.zarr`s have the same time
extent and shape, so it is the same movie; (3) `estimated_number_of_nodes` far exceeds the
annotated node count; (4) `claude_fork`'s output gives a real adjustment factor.

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
TRAIN, TEST = ROOT / "train", ROOT / "test"

test_stems = sorted(p.name.split(".")[0] for p in TEST.glob("*.zarr"))
train_zarr = sorted(p.name.split(".")[0] for p in TRAIN.glob("*.zarr"))
train_geff = sorted(p.name.split(".")[0] for p in TRAIN.glob("*.geff"))

print(f"test  : {len(test_stems)} zarr   {test_stems}")
print(f"train : {len(train_zarr)} zarr, {len(train_geff)} geff")
print(f"train prefixes: {sorted({s.split('_')[0] for s in train_zarr})}")
print(f"test  prefixes: {sorted({s.split('_')[0] for s in test_stems})}")

overlap_geff = [s for s in test_stems if s in set(train_geff)]
overlap_zarr = [s for s in test_stems if s in set(train_zarr)]
print(f"\ntest stems with a train .geff : {overlap_geff}")
print(f"test stems with a train .zarr : {overlap_zarr}")

In [ ]:
# Same movie, or same hash on a different crop? Compare the zarr metadata directly.
def zmeta(p):
    out = {}
    j = p / "zarr.json"
    if j.exists():
        d = json.loads(j.read_text())
        out["node_type"] = d.get("node_type")
        out["attrs_keys"] = sorted((d.get("attributes") or {}).keys())
    # the pixel array lives under `0`; its own zarr.json carries the shape
    a = p / "0" / "zarr.json"
    if a.exists():
        d = json.loads(a.read_text())
        out["shape"] = d.get("shape")
        out["dtype"] = d.get("data_type")
        out["chunks"] = ((d.get("chunk_grid") or {}).get("configuration") or {}).get(
            "chunk_shape")
    return out

for s in test_stems:
    t = zmeta(TEST / f"{s}.zarr")
    r = zmeta(TRAIN / f"{s}.zarr") if (TRAIN / f"{s}.zarr").exists() else None
    same = (r is not None and t.get("shape") == r.get("shape"))
    print(f"{s:<18} test shape={t.get('shape')}  train shape="
          f"{r.get('shape') if r else '(absent)'}   same={same}")

In [ ]:
# estimated_number_of_nodes and the annotated node count, from the train GEFF.
def geff_meta(stem):
    p = TRAIN / f"{stem}.geff"
    if not p.exists():
        return None
    d = json.loads((p / "zarr.json").read_text())
    g = ((d.get("attributes") or {}).get("geff") or {})
    est = None
    stack = [d]
    while stack:
        o = stack.pop()
        if isinstance(o, dict):
            for k, v in o.items():
                if "estimated" in str(k) and "node" in str(k):
                    est = v
                stack.append(v)
        elif isinstance(o, list):
            stack.extend(o)
    axes = {a.get("name"): (a.get("min"), a.get("max")) for a in (g.get("axes") or [])}
    # node count: the ids array's own zarr.json shape
    n_ann = None
    for cand in ("nodes/ids/zarr.json", "nodes/props/t/values/zarr.json"):
        c = p / cand
        if c.exists():
            n_ann = json.loads(c.read_text()).get("shape")
            break
    return {"est": est, "t_range": axes.get("t"), "n_annotated": n_ann}

rows = {}
for s in test_stems:
    m = geff_meta(s)
    rows[s] = m
    print(f"{s:<18} {m}")

In [ ]:
FORK_NODES = {"6bba_05db0fb1": 69112, "44b6_0113de3b": 25425,
              "44b6_0b24845f": 18089, "6bba_05b6850b": 6033}

print("=" * 78)
print("PREDICTION GRADING")
print("=" * 78)

ok1 = bool(overlap_geff)
print(f"\n1. a test stem has a train .geff  ->  {'PASS' if ok1 else 'FAIL'}")
print(f"   {overlap_geff or 'none'}")

same_shape = []
for s in overlap_zarr:
    t = zmeta(TEST / f"{s}.zarr").get("shape")
    r = zmeta(TRAIN / f"{s}.zarr").get("shape")
    same_shape.append(t == r)
ok2 = bool(same_shape) and all(same_shape)
print(f"\n2. the colliding zarrs are the same movie  ->  "
      f"{'PASS' if ok2 else 'FAIL' if same_shape else 'NOT GRADED (no zarr overlap)'}")
if not ok2 and same_shape:
    print("   Same name, different pixels: the organisers re-cropped and the collision is")
    print("   a hash coincidence. notes/49 stands and nothing below is usable.")

ests = {s: m["est"] for s, m in rows.items() if m and isinstance(m.get("est"), (int, float))}
ok3 = False
if ests:
    print("\n3. N_est far exceeds the annotated node count")
    for s, e in ests.items():
        n = rows[s].get("n_annotated")
        n0 = n[0] if isinstance(n, list) and n else None
        ratio = (e / n0) if n0 else float("nan")
        print(f"   {s:<18} est {e:>10,.0f}   annotated {str(n0):>8}   {ratio:>7.1f}x")
        ok3 = ok3 or (n0 is not None and e > 3 * n0)
    print(f"   ->  {'PASS' if ok3 else 'FAIL'}")
else:
    print("\n3. NOT GRADED — no estimated_number_of_nodes read")

if ests:
    print("\n4. claude_fork's adjustment factor on the datasets we can now see")
    print(f"   {'dataset':<18}{'N_pred':>10}{'N_est':>12}{'ratio':>10}{'factor':>10}")
    tp = te = 0.0
    for s in sorted(ests):
        n, e = FORK_NODES[s], ests[s]
        tp += n; te += e
        r = (n - e) / e
        print(f"   {s:<18}{n:>10,}{e:>12,.0f}{r:>+10.3f}{1 - 0.1 * r:>10.4f}")
    R = (tp - te) / te
    print(f"   {'subtotal':<18}{tp:>10,.0f}{te:>12,.0f}{R:>+10.3f}{1 - 0.1 * R:>10.4f}")
    print(f"\n   At edge_J ~ 0.92 that factor is worth {0.92 * (1 - 0.1 * R) - 0.92:+.4f}.")
    if R > 0:
        print("   OVER budget: pruning toward N_est pays twice -- a larger factor AND")
        print("   fewer false edges. This is the first unclaimed term we have found.")
    else:
        print("   UNDER budget already: the bonus is being collected, and further pruning")
        print("   trades edge_J against a factor above 1. Prices the direction near zero.")
print("=" * 78)